# Qwen3-VL-8B-Instruct-GGUF Inference

This notebook runs inference on the Unsloth GGUF model using `llama-cpp-python`.

### Setup
1. Ensure `llama-cpp-python` is installed.
2. Ensure you have the `dentex_loader.py` in the same directory.

In [ ]:
# Install dependencies if needed (uncomment to run)
# !pip install llama-cpp-python huggingface_hub pillow numpy

In [ ]:
import os
import base64
from io import BytesIO
from PIL import Image
from huggingface_hub import hf_hub_download, list_repo_files
from llama_cpp import Llama

# Dental Dataset Loader
try:
    from dentex_loader import get_dentex_dataset
    HAS_DATASET = True
except ImportError:
    HAS_DATASET = False
    print("dentex_loader not found, using dummy image if available.")

In [ ]:
# Model Configuration
REPO_ID = 'unsloth/Qwen3-VL-8B-Instruct-GGUF'
MODEL_FILTER = 'q4_k_m.gguf' # Look for 4-bit quantization

print(f"Searching for {MODEL_FILTER} in {REPO_ID}...")
files = list_repo_files(REPO_ID)
candidates = [f for f in files if f.endswith('.gguf') and 'q4_k_m' in f.lower()]

if not candidates:
    candidates = [f for f in files if f.endswith('.gguf')]
    print("Preferred quantization not found, falling back to:", candidates[0])

selected_model = candidates[0]
print(f"Downloading/Loading: {selected_model}")

model_path = hf_hub_download(repo_id=REPO_ID, filename=selected_model)

In [ ]:
# Initialize Llama
# Ensure n_gpu_layers is set appropriately for your GPU (e.g. -1 for all layers)
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_gpu_layers=-1, 
    verbose=True
)

In [ ]:
def image_to_base64_data_uri(image):
    buffered = BytesIO()
    # Resize if image is too large for VLM context
    if image.width > 1024 or image.height > 1024:
        image.thumbnail((1024, 1024))
    image.save(buffered, format='JPEG')
    img_str = base64.b64encode(buffered.getvalue()).decode('utf-8')
    return f'data:image/jpeg;base64,{img_str}'

def run_inference(image, prompt='Describe the condition appearing in this dental X-ray.'):
    uri = image_to_base64_data_uri(image)
    
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'text', 'text': prompt},
                {'type': 'image_url', 'image_url': {'url': uri}}
            ]
        }
    ]
    
    response = llm.create_chat_completion(
        messages=messages,
        max_tokens=256,
        temperature=0.2
    )
    return response['choices'][0]['message']['content']

In [ ]:
# Run Test
if HAS_DATASET:
    ds = get_dentex_dataset(split='train')
    print(f"Dataset loaded with {len(ds)} samples.")
    
    # limit to first 3
    for i in range(3):
        item = ds[i]
        img = item['image']
        print(f"\n--- Processing Image {i} ---")
        res = run_inference(img)
        print("Result:", res)
        # Optionally display image
        # display(img)
else:
    print("No dataset found. Please provide an image to test.")